<a href="https://colab.research.google.com/github/sanjana0422/solar/blob/main/Solar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#loading dataset
!pip install elpv-dataset -q
from elpv_dataset.utils import load_dataset
images,proba,types=load_dataset()
print(images.shape)

(2624, 300, 300)


In [ ]:
#binary classification
import numpy as np

proba_rounded = np.round(proba, 2)

def count(p):
    return np.sum(np.isclose(proba, p, atol=0.05))

print(f"Types: mono={np.sum(types=='mono')}, poly={np.sum(types=='poly')}")
print(f"Defect proba: 0.0={count(0.0)}, 0.33={count(0.33)}, 0.66={count(0.66)}, 1.0={count(1.0)}")
print(f"Total: {len(proba)} = 1508+295+106+715 = {1508+295+106+715}")

y_binary = (proba >= 0.5).astype(int)
print(f"\nBinary mapping: Functional (<0.5)={np.sum(y_binary==0)}, Defective (>=0.5)={np.sum(y_binary==1)}")

Types: mono=1074, poly=1550
Defect proba: 0.0=1508, 0.33=295, 0.66=106, 1.0=715
Total: 2624 = 1508+295+106+715 = 2624

Binary mapping: Functional (<0.5)=1803, Defective (>=0.5)=821


In [ ]:
#preprocessing
import cv2, torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

# 1. Normalization + 2. Cropping + 3. Augmentation together
train_tf = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.3),
    T.ToTensor()
])

class ELPVDataset(Dataset):
    def __init__(self, imgs, labels, transform=None):
        self.imgs = imgs
        self.labels = (labels >= 0.5).astype(float)
        self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = self.imgs[i]
        if img.max() <= 1.0: img = (img*255).astype('uint8')
        else: img = img.astype('uint8')
        img = cv2.equalizeHist(img) # NORMALIZATION
        img = cv2.resize(img, (128,128)) # CROPPING/RESIZING
        if self.transform:
            img = self.transform(img) # AUGMENTATION
        else:
            img = torch.tensor(img/255., dtype=torch.float32).unsqueeze(0)
        return img, torch.tensor(self.labels[i], dtype=torch.float32)
#  spliting for custom cnn
X_temp, X_test, y_temp, y_test = train_test_split(images, proba, test_size=0.2, random_state=42, stratify=(proba>=0.5))
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=(y_temp>=0.5))

train_loader = DataLoader(ELPVDataset(X_train, y_train, train_tf), batch_size=32, shuffle=True)
val_loader = DataLoader(ELPVDataset(X_val, y_val, None), batch_size=32)
test_loader = DataLoader(ELPVDataset(X_test, y_test, None), batch_size=32)

print(f"Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}")

Train:1836 Val:263 Test:525


In [ ]:
#custom cnn designed,trained,tuned
import torch.nn as nn
from sklearn.metrics import roc_auc_score

class SolarDefectCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16,256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256,1)
        )
    def forward(self,x): return self.classifier(self.features(x)).squeeze(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SolarDefectCNN().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.2]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best=0
for epoch in range(12):
    model.train()
    for xb,yb in train_loader:
        xb,yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
    model.eval(); preds=[]
    with torch.no_grad():
        for xb,_ in val_loader:
            preds.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    auc = roc_auc_score((y_val>=0.5).astype(int), preds)
    print(f"Epoch {epoch+1} Val AUC={auc:.4f}")
    if auc>best:
        best=auc
        torch.save(model.state_dict(), "solar_defect_cnn_custom.pth")

print(f"Saved solar_defect_cnn_custom.pth with AUC {best:.4f}")

Epoch 1 Val AUC=0.7512
Epoch 2 Val AUC=0.7137
Epoch 3 Val AUC=0.7567
Epoch 4 Val AUC=0.7643
Epoch 5 Val AUC=0.7853
Epoch 6 Val AUC=0.7923
Epoch 7 Val AUC=0.7916
Epoch 8 Val AUC=0.7862
Epoch 9 Val AUC=0.8034
Epoch 10 Val AUC=0.7994
Epoch 11 Val AUC=0.7978
Epoch 12 Val AUC=0.8053
Saved solar_defect_cnn_custom.pth with AUC 0.8053


In [ ]:
#evalution
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt, numpy as np

model.load_state_dict(torch.load("solar_defect_cnn_custom.pth", map_location=device))
model.eval()
y_true, y_pred_prob = [], []
with torch.no_grad():
    for xb,yb in test_loader:
        y_true.extend(yb.numpy())
        y_pred_prob.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())

y_true = (np.array(y_true)>=0.5).astype(int)
y_pred = (np.array(y_pred_prob)>=0.5).astype(int)

print(f"Accuracy: {accuracy_score(y_true,y_pred):.4f}")
print(f"Precision: {precision_score(y_true,y_pred):.4f}")
print(f"Recall: {recall_score(y_true,y_pred):.4f}")
print(f"F1: {f1_score(y_true,y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_true,y_pred_prob):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_true,y_pred))

Accuracy: 0.7448
Precision: 0.5824
Recall: 0.6463
F1: 0.6127
ROC-AUC: 0.7806
Confusion Matrix:
 [[285  76]
 [ 58 106]]
